# ML Homework 5 Guide

1. Save a copy of this ipynb file in your GoogleDrive or PC.
2. Edit the name of this code from "HW5.ipynb" to "HW5_(your name).ipynb"
3. Fill out the code cells below according to descriptions.
4. Save and upload to BrightSpace. (DO NOT clear the outputs of your code)
5. Convert the .ipynb file to PDF file and upload together



** 6. Upload the saved 'Original vs Reconstructed Data' image files for both the worst and best models.

** 7. Upload the saved 'Latent Space Visualization (t-SNE)' image file for both the worst and best models.


# [HW 5] Grid Search for ConvAE and Visualization

1. Load your 'SoundSTFT.npy' file as you loaded in the 'ML10_Code1'.
2. Prepare training and test dataset with the test data ratio of """10%""".
3. Perform a grid search to find the best dimension of bottle neck layer for the ConvAE model.
  - You don't need to search other hyperparameters.

4. Determine your best and worst CNN model based on reconstruction error.
5. Compare original and reconstructed STFT spectrogram for both the worst and best models and save them as image files (.png or .jpg).
  - Upload the comparison images to BrightSpace with ipynb and pdf files

6. Extract latent space features from both the worst and best models and visualize them using t-SNE.
  - Save and upload the visualization results to BrightSpace as well.

### Install & Imports

In [ ]:
import os, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sp
from scipy import signal

import librosa
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow:', tf.__version__)
print('librosa   :', librosa.__version__)

### Data Loading, Indexing & Segmentation

In [ ]:
# Download the industrial sound recording
file_url = (
    'https://github.com/purduelamm/purdue_me597_iiot/blob/main/'
    'ml_tutorial/Dataset_IndustrialSound/Industrial_data_Stethoscope.wav?raw=true'
)
file_path = 'Industrial_data_Stethoscope.wav'
urllib.request.urlretrieve(file_url, file_path)

sound, rate = librosa.load(file_path, sr=None)

CLASS_NAMES = ['Tool Change', 'Chip Conveyer',
               'Moving X', 'Moving Y', 'Moving Z', 'Spindle']
NUM_CLASS   = len(CLASS_NAMES)

timestamps  = [12.46, 32.51, 65.57, 95.30, 104.36, 112.12]
indices     = np.multiply(rate, timestamps).astype(np.int64)

# Split into class segments
raw_classes = []
for i in range(NUM_CLASS):
    if i == NUM_CLASS - 1:
        raw_classes.append(sound[indices[i]:])
    else:
        raw_classes.append(sound[indices[i]:indices[i+1]])

# Fixed-length segmentation: 1-second window, 0.8-second overlap
SEGMENT_LEN = int(1.0 * rate)
OVERLAP     = int(0.8 * rate)
STEP        = SEGMENT_LEN - OVERLAP

def segmentation(data):
    segs, start = [], 0
    while start + SEGMENT_LEN <= len(data):
        segs.append(data[start: start + SEGMENT_LEN])
        start += STEP
    return np.array(segs, dtype=np.float32)

segments_per_class = [segmentation(c) for c in raw_classes]


### Prepare Training and Test Dataset

In [ ]:
TEST_RATIO = 0.10 # Test data ratio as per instruction 2
VAL_RATIO  = 0.15   # fraction of training data held out for validation
RANDOM_STATE = 42

train_segs, test_segs = [], []
train_labels_list, test_labels_list = [], []

for class_idx, segs in enumerate(segments_per_class):
    tr, te = train_test_split(segs, test_size=TEST_RATIO,
                              random_state=RANDOM_STATE)
    train_segs.append(tr);  test_segs.append(te)
    train_labels_list.extend([class_idx] * len(tr))
    test_labels_list.extend([class_idx]  * len(te))

X_train_raw = np.concatenate(train_segs, axis=0)
X_test_raw  = np.concatenate(test_segs,  axis=0)
y_train     = np.array(train_labels_list)
y_test      = np.array(test_labels_list)

# ── Stratified validation split ───────────────────────────────────────────
# keras validation_split takes the LAST N% of the array before shuffling.
# Because classes are concatenated in order, that slice is dominated by the
# last class, causing biased validation metrics (val_acc starts at 1.0 then
# collapses for the CNN).  A stratified split guarantees every class appears
# at the correct proportion in both halves — regardless of model or epoch.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train,
    test_size    = VAL_RATIO,
    stratify     = y_train,       # <── key: preserves class proportions
    random_state = RANDOM_STATE,
)

print(f'Train : {X_train_raw.shape[0]} segments  class dist: {np.bincount(y_train)}')
print(f'Val   : {X_val_raw.shape[0]}  segments  class dist: {np.bincount(y_val)}')
print(f'Test  : {X_test_raw.shape[0]}  segments  class dist: {np.bincount(y_test)}')

# One-hot labels
y_train_oh = tf.keras.utils.to_categorical(y_train, NUM_CLASS)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   NUM_CLASS)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  NUM_CLASS)

### CNN 

In [ ]:
def CNN_model(input_data):
    keras.backend.clear_session()

    model = keras.Sequential()
    model.add(
        keras.layers.InputLayer(
            input_shape=(
                input_data.shape[1],
                input_data.shape[2],
                input_data.shape[3],
            )
        )
    )  # Input layer

    model.add(
        keras.layers.Conv2D(
            filters=2,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="same",
            activation="relu",
        )
    )  # Convolution layer 1
    model.add(
        keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))
    )  # Pooling layer 1
    model.add(
        keras.layers.Conv2D(
            filters=4,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="same",
            activation="relu",
        )
    )  # Convolution layer 2
    model.add(
        keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))
    )  # Pooling layer 2

    model.add(keras.layers.Flatten())  # Flatten layer
    model.add(keras.layers.Dense(units=10, activation="relu"))  # Dense layer

    model.add(
        keras.layers.Dense(units=2, activation="softmax")
    )  # Output Layer

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learningRate),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model

## ML7 and ML8 Summary and Deliverables

Answer the following questions for your achievements

### Q1. Please summarize ML7 and ML8.

---

Write down A1 here.

---

### Q2. What skills did you have to develop to accomplish this project?

---

Wirte down A2 here.

---

### Q3. What aspects of this project were the most beneficial for your learning?

---

Wirte down A3 here.

---

### Q4. What challenges did you encounter in completing the project?

---

Wirte down A4 here.

---

### Q5. How did you overcome the challenges or remedy the problems encountered?

---

Wirte down A5 here.

---